# Vahan Dataset — Data Cleaning & Preparation

This section documents the data-cleaning and validation steps performed on the Vahan vehicle registration dataset before analysis and visualization.

Each step below explains the purpose of the code and the data-quality issue being checked or corrected.

# 1. Load the raw dataset

The Vahan vehicle registration dataset is loaded from the CSV file into a pandas DataFrame named `Vahan_dataset`.

The first few rows are displayed to understand the structure of the data and confirm that the dataset has been loaded correctly.

In [37]:
import numpy as numpy
import pandas as pd

Vahan_dataset= pd.read_csv("C:/Users/Admin/Documents/India_VAHAN_Dataset.csv")

Vahan_dataset.head(10)

,Registration_Number,Registration_Date,Registration_Year,State,RTO_Office,Vehicle_Category,Vehicle_Sub_Type,Manufacturer_Brand,Fuel_Type,Emission_Norm,Engine_CC,Seating_Capacity,Vehicle_Age_Years
0,REG100001,12/13/2019,2019,Madhya Pradesh,Bhopal (MP-04),2W,Scooter,Bajaj,Petrol,BS4,400,2,5
1,REG100002,1/14/2020,2020,Karnataka,Belgaum (KA-22),2W,Motorcycle,Honda,Petrol,BS6,400,2,4
2,REG100003,3/23/2023,2023,Karnataka,Delhi South (DL-09),2W,Scooter,Ola Electric,Petrol,BS6,250,2,1
3,REG100004,6/26/2021,2021,Karnataka,Kota (RJ-20),2W,Scooter,Honda,Petrol,BS6,0,2,3
4,REG100005,11/13/2022,2022,Karnataka,Ahmedabad (GJ-01),4W,Hatchback,Honda,Petrol,BS6,1598,6,2
5,REG100006,7/13/2024,2024,Karnataka,Thiruvananthapuram (KL-01),2W,Motorcycle,Royal Enfield,Petrol,BS6,160,2,0
6,REG100007,11/28/2021,2021,Karnataka,Durgapur (WB-40),2W,Scooter,Bajaj,CNG,BS6,110,2,3
7,REG100008,12/19/2021,2021,Karnataka,Ludhiana (PB-10),4W,Sedan,Tata Motors,Diesel,BS6,0,6,3
8,REG100009,5/18/2023,2023,Karnataka,Gwalior (MP-07),LCV,Mini Truck,Mahindra,Diesel,BS6,2523,9,1
9,REG100010,3/13/2022,2022,Karnataka,Amritsar (PB-02),2W,Scooter,Suzuki,EV,BS6,0,2,2


## 2. Check data quality: duplicates and missing values

Before making any changes, the dataset is checked for:

- Duplicate rows using `duplicated()`.
- Missing/null values in each column using `isnull().sum()`.

These checks help identify basic data-quality issues that may need to be addressed before analysis.

In [39]:
Vahan_dataset.duplicated().sum()

Vahan_dataset.isnull().sum()

Registration_Number    0
Registration_Date      0
Registration_Year      0
State                  0
RTO_Office             0
Vehicle_Category       0
Vehicle_Sub_Type       0
Manufacturer_Brand     0
Fuel_Type              0
Emission_Norm          0
Engine_CC              0
Seating_Capacity       0
Vehicle_Age_Years      0
dtype: int64

## 3. Inspect data types and registration dates

The data types of all columns are reviewed using `dtypes`.

The `Registration_Date` column is also summarized using `describe()` to understand its values and distribution and to check that the registration-date information looks reasonable.

In [40]:
Vahan_dataset.dtypes

Vahan_dataset["Registration_Date"].describe()

count          3700
unique         1861
top       6/11/2024
freq              8
Name: Registration_Date, dtype: object

## 4. Inspect categorical values for consistency

The unique values of the main categorical columns are reviewed:

- `Fuel_Type`
- `Manufacturer_Brand`
- `Vehicle_Category`
- `Vehicle_Sub_Type`
- `Emission_Norm`

This step helps identify inconsistent, unexpected, or incorrectly classified category values before creating visualizations.

In [41]:
for col in [
    "Fuel_Type",
    "Manufacturer_Brand",
    "Vehicle_Category",
    "Vehicle_Sub_Type",
    "Emission_Norm"
]:
    print(col)
    print(Vahan_dataset[col].unique())

Fuel_Type
<ArrowStringArray>
['Petrol', 'CNG', 'Diesel', 'EV', 'Hybrid']
Length: 5, dtype: str
Manufacturer_Brand
<ArrowStringArray>
[        'Bajaj',         'Honda',  'Ola Electric', 'Royal Enfield',
   'Tata Motors',      'Mahindra',        'Suzuki',         'Skoda',
        'Toyota',           'Kia',          'Hero',        'Yamaha',
           'KTM',       'Piaggio',       'Renault',        'Eicher',
          'TAFE',            'MG',       'Hyundai',  'Force Motors',
           'TVS',           'MAN',     'Atul Auto', 'Kinetic Green',
    'John Deere',         'Ather',       'Escorts',    'BharatBenz',
 'Maruti Suzuki',         'Volvo', 'Ashok Leyland',      'Sonalika']
Length: 32, dtype: str
Vehicle_Category
<ArrowStringArray>
['2W', '4W', 'LCV', 'OTH', '3W', 'HCV']
Length: 6, dtype: str
Vehicle_Sub_Type
<ArrowStringArray>
[      'Scooter',    'Motorcycle',     'Hatchback',         'Sedan',
    'Mini Truck',           'MUV',       'Tractor',         'Moped',
           'SUV',   

## 5. Correct fuel type and engine capacity for Ola Electric

A data correction is applied for **Ola Electric** records.

- Records for Ola Electric with `Engine_CC = 0` and an incorrect fuel type such as Petrol, Diesel, Hybrid, or CNG are reclassified as `EV`.
- `Engine_CC` for Ola Electric is explicitly set to `0`, because these vehicles are electric and do not have a conventional engine displacement value.

A count is then checked to confirm that the targeted inconsistent records have been corrected.

In [42]:
#Replacing other fuel types with EV for Ola Electric

Vahan_dataset.loc[
      (Vahan_dataset['Engine_CC'] == 0) & (Vahan_dataset['Manufacturer_Brand']=="Ola Electric")
    & (Vahan_dataset['Fuel_Type'].isin(["Petrol", "Diesel","Hybrid","CNG"])),"Fuel_Type"]= "EV"

#Replacing Engine_CC for Ola Electric to 0
Vahan_dataset.loc[
      (Vahan_dataset['Manufacturer_Brand']=="Ola Electric")
      ,"Engine_CC"]= 0

len(Vahan_dataset[(Vahan_dataset['Engine_CC'] != 0) & (Vahan_dataset['Manufacturer_Brand']=="Ola Electric")
    & (Vahan_dataset['Fuel_Type'].isin(["Petrol", "Diesel","Hybrid","CNG"]))])

0

## 6. Validate the Ola Electric fuel-type correction

The `Fuel_Type` values for Ola Electric are reviewed after the correction.

This validation confirms that Ola Electric records now have the expected fuel-type classification.

In [43]:
#Checking the values of Fuel_Type for Ola Electric
Vahan_dataset[Vahan_dataset["Manufacturer_Brand"]=='Ola Electric']["Fuel_Type"].unique()

<ArrowStringArray>
['Petrol', 'EV', 'CNG', 'Hybrid']
Length: 4, dtype: str

## 7. Identify zero-engine-capacity records that are not EVs

Records where `Engine_CC` is `0` but `Fuel_Type` is not `EV` are identified.

These records require further investigation because a zero engine capacity may indicate missing or incorrect engine-capacity data for non-electric vehicles.

In [44]:
Vahan_dataset[(Vahan_dataset["Engine_CC"]==0) & (~Vahan_dataset["Fuel_Type"].isin (['EV'])) ]

,Registration_Number,Registration_Date,Registration_Year,State,RTO_Office,Vehicle_Category,Vehicle_Sub_Type,Manufacturer_Brand,Fuel_Type,Emission_Norm,Engine_CC,Seating_Capacity,Vehicle_Age_Years
2,REG100003,3/23/2023,2023,Karnataka,Delhi South (DL-09),2W,Scooter,Ola Electric,Petrol,BS6,0,2,1
3,REG100004,6/26/2021,2021,Karnataka,Kota (RJ-20),2W,Scooter,Honda,Petrol,BS6,0,2,3
7,REG100008,12/19/2021,2021,Karnataka,Ludhiana (PB-10),4W,Sedan,Tata Motors,Diesel,BS6,0,6,3
26,REG100027,6/10/2022,2022,Karnataka,Coimbatore (TN-37),2W,Motorcycle,Ola Electric,CNG,BS6,0,2,2
33,REG100034,12/18/2023,2023,Karnataka,Belgaum (KA-22),2W,Scooter,Ola Electric,Petrol,BS6,0,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3679,REG103680,5/8/2023,2023,Punjab,Patiala (PB-11),3W,E-Rickshaw,Kinetic Green,CNG,BS6,0,6,1
3681,REG103682,1/21/2023,2023,Maharashtra,Nashik (MH-15),LCV,Pickup Van,Tata Motors,Diesel,BS6,0,9,1
3688,REG103689,7/5/2018,2018,Kerala,Kozhikode (KL-11),2W,Moped,Ola Electric,Hybrid,BS3,0,2,6
3692,REG103693,6/16/2020,2020,Tamil Nadu,Chennai Central (TN-01),2W,Moped,Ola Electric,Petrol,BS6,0,2,4


## 8. Impute missing/zero engine capacity for non-EV vehicles

For non-EV vehicles where `Engine_CC` is `0`, the value is replaced using the **median engine capacity for the corresponding `Manufacturer_Brand` and `Fuel_Type` group**.

The imputation is performed only when a positive engine-capacity value is available within that manufacturer/fuel-type group. If no positive value exists, the value remains `0`.

This provides a group-specific estimate instead of applying one overall median to all vehicles.

In [45]:
Vahan_dataset['Engine_CC'] = (
    Vahan_dataset.groupby(['Manufacturer_Brand', 'Fuel_Type'])['Engine_CC']
    .transform(
        lambda x: x.mask(
            (x == 0) &
            (Vahan_dataset.loc[x.index, 'Fuel_Type'] != 'EV'),
            x[x > 0].median() if not x[x > 0].empty else 0
        )
    )
)


## 9. Validate the engine-capacity imputation

The dataset is checked again for non-EV records where `Engine_CC` is still `0`.

This confirms whether any zero engine-capacity values remain after the imputation step and identifies records that may require additional investigation.

In [46]:
Vahan_dataset[
    (Vahan_dataset['Fuel_Type'] != 'EV') &
    (Vahan_dataset['Engine_CC'] == 0)
]

,Registration_Number,Registration_Date,Registration_Year,State,RTO_Office,Vehicle_Category,Vehicle_Sub_Type,Manufacturer_Brand,Fuel_Type,Emission_Norm,Engine_CC,Seating_Capacity,Vehicle_Age_Years
2,REG100003,3/23/2023,2023,Karnataka,Delhi South (DL-09),2W,Scooter,Ola Electric,Petrol,BS6,0.0,2,1
26,REG100027,6/10/2022,2022,Karnataka,Coimbatore (TN-37),2W,Motorcycle,Ola Electric,CNG,BS6,0.0,2,2
33,REG100034,12/18/2023,2023,Karnataka,Belgaum (KA-22),2W,Scooter,Ola Electric,Petrol,BS6,0.0,2,1
65,REG100066,7/22/2024,2024,Karnataka,Belgaum (KA-22),2W,Scooter,Ola Electric,CNG,BS6,0.0,2,0
85,REG100086,8/13/2023,2023,Karnataka,Bangalore Central (KA-01),2W,Scooter,Ola Electric,Hybrid,BS6,0.0,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3644,REG103645,10/12/2022,2022,Delhi,Delhi West (DL-08),2W,Scooter,Ola Electric,Petrol,BS6,0.0,2,2
3651,REG103652,4/3/2021,2021,Uttar Pradesh,Varanasi (UP-65),2W,Motorcycle,Ola Electric,CNG,BS6,0.0,2,3
3665,REG103666,3/2/2020,2020,Delhi,Delhi South (DL-09),2W,Scooter,Ola Electric,Petrol,BS6,0.0,2,4
3688,REG103689,7/5/2018,2018,Kerala,Kozhikode (KL-11),2W,Moped,Ola Electric,Hybrid,BS3,0.0,2,6


## 10. Check unusually high engine-capacity values

Vehicle sub-types associated with `Engine_CC > 2000` are reviewed.

This is a validation step to identify potentially unusual engine-capacity values or vehicle classifications that could affect the analysis. The code does not automatically remove these records.

In [47]:
Vahan_dataset[Vahan_dataset["Engine_CC"]>2000]["Vehicle_Sub_Type"].unique()

<ArrowStringArray>
[      'Sedan',  'Mini Truck',         'MUV',     'Tractor',         'Bus',
 'Road Roller',   'Light Bus',      'Tanker',     'Trailer',  'Pickup Van',
         'SUV',       'Crane',   'Hatchback',       'Truck',  'E-Rickshaw']
Length: 15, dtype: str

## 11. Validate E-Rickshaw records

E-Rickshaw records are inspected together with their manufacturer, vehicle subtype, fuel type, and engine capacity.

This helps verify that the fuel-type and engine-capacity classifications for E-Rickshaws are reasonable and consistent with the intended dataset structure.

In [48]:
Vahan_dataset[
    (Vahan_dataset["Vehicle_Sub_Type"] == "E-Rickshaw") 
][["Manufacturer_Brand", "Vehicle_Sub_Type", "Fuel_Type", "Engine_CC"]]

,Manufacturer_Brand,Vehicle_Sub_Type,Fuel_Type,Engine_CC
97,Atul Auto,E-Rickshaw,Petrol,225.0
138,Mahindra,E-Rickshaw,CNG,250.0
150,Atul Auto,E-Rickshaw,EV,0.0
162,Piaggio,E-Rickshaw,CNG,200.0
170,Bajaj,E-Rickshaw,EV,0.0
...,...,...,...,...
3643,Mahindra,E-Rickshaw,Diesel,2523.0
3656,Atul Auto,E-Rickshaw,EV,0.0
3659,Kinetic Green,E-Rickshaw,Diesel,225.0
3679,Kinetic Green,E-Rickshaw,CNG,225.0


## 12. Perform final data-type and date checks

The data types and `Registration_Date` summary are reviewed again after the cleaning steps.

This acts as a final validation that the cleaning operations have not introduced unexpected data-type or date-related issues.

In [49]:
Vahan_dataset.dtypes

Vahan_dataset["Registration_Date"].describe()

count          3700
unique         1861
top       6/11/2024
freq              8
Name: Registration_Date, dtype: object

## 13. Validate vehicle age calculation

The calculated `Vehicle_Age_Years` is compared against:

`2024 - Registration_Year`

Any records where the two values do not match are displayed for investigation.

This ensures that the vehicle-age field is consistent with the registration year used in the analysis.

In [50]:
#chceking the Vehicle Age
Vahan_dataset[
    Vahan_dataset["Vehicle_Age_Years"] !=
    (2024 - Vahan_dataset["Registration_Year"])
][["Registration_Year", "Vehicle_Age_Years"]]

,Registration_Year,Vehicle_Age_Years


## 14. Export the cleaned dataset

After the data-quality checks and corrections are completed, the cleaned DataFrame is exported to an Excel file named `Vahan_dataset_cleaned.xlsx`.

The cleaned file can then be used as the source for downstream analysis and Power BI visualizations.

In [51]:
Vahan_dataset.to_excel(
    r"C:\Users\Admin\Documents\Vahan_dataset_cleaned.xlsx",
    index=False
)